In [1]:
%cd /Users/akarsh/Documents/projects/fastInfer/StoryForge

/Users/akarsh/Documents/projects/fastInfer/StoryForge


In [8]:
import math
import numpy as np
import torch

from storyforge.config import load_config
from storyforge.model import ForgeGPT
from storyforge.optimization import (
    build_optimizer,
    learning_rate_for_step,
)
from storyforge.training import train_update

In [9]:
config = load_config("configs/default.toml")

In [10]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [11]:
checkpoint = torch.load(
    config["checkpoints"]["initial_path"],
    map_location="cpu",
    weights_only=True,
)

In [12]:
torch.manual_seed(config["seed"])

In [13]:
pilot_model = ForgeGPT(
    checkpoint["config"],
    vocab_size=checkpoint["vocab_size"],
)

In [14]:
pilot_model.load_state_dict(checkpoint["model"])
pilot_model = pilot_model.to(device)

In [15]:
pilot_optimizer = build_optimizer(pilot_model, config)

In [16]:
train_tokens = np.load(
    config["encoding"]["train_path"],
    mmap_mode="r",
)

In [17]:
rng = np.random.default_rng(config["seed"])

In [18]:
tokens_per_update = (
    config["training"]["batch_size"]
    * config["model"]["max_context"]
    * config["training"]["gradient_accumulation_steps"]
)

In [19]:
total_steps = math.ceil(
    config["training"]["token_budget"] / tokens_per_update
)

In [20]:
print("Device:", device)
print("Tokens per update:", tokens_per_update)
print("Planned main-run updates:", total_steps)

Device: mps
Tokens per update: 16384
Planned main-run updates: 1526


In [21]:
lr = learning_rate_for_step(
    step=1,
    total_steps=total_steps,
    config=config,
)


In [22]:
for group in pilot_optimizer.param_groups:
    group["lr"] = lr

In [23]:
result = train_update(
    model=pilot_model,
    optimizer=pilot_optimizer,
    train_tokens=train_tokens,
    config=config,
    rng=rng,
    device=device,
)


In [24]:
print("Learning rate:", lr)
print("Training loss:", result["train_loss"])
print("Processed target tokens:", result["processed_tokens"])


Learning rate: 2.9999999999999997e-06
Training loss: 9.072906732559204
Processed target tokens: 16384


In [25]:
assert math.isfinite(result["train_loss"])
assert result["processed_tokens"] == tokens_per_update